# KUL CV GA2 — EfficientNet-V2-S 320 (Kaggle)

Backbone: `efficientnet_v2_s` (ImageNet-1K) · Input: 320×320 · Loss: AsymmetricLoss · TTA ✓ · Stage-3 全量重训 ✓

**EfficientNet-V2-S 概览**
- 参数量 ~21.5M（比 ConvNeXt-Small 小一半）
- 特征维度 1280（`model.features` + `model.avgpool` 后 flatten）
- ImageNet-1K top-1: 84.2%（比 EfficientNet-B3 高）
- 适合显存受限场景：batch_size=16 在 T4 上安全

**使用前**：在 Kaggle Notebook 的 **Add Data** 中加入 `kul-computer-vision-ga-2-2026` 数据集。

In [ ]:
from pathlib import Path
import gc, os, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

print('torch:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    !nvidia-smi
    torch.backends.cudnn.benchmark = True

In [ ]:
# ── 配置 ──────────────────────────────────────────────────────────────────────
EXPERIMENT   = 'efficientnet_v2_s_320'
BACKBONE     = 'efficientnet_v2_s'
FEAT_DIM     = 1280   # EfficientNet-V2-S features+avgpool 输出维度
IMG_SIZE     = 320
BATCH_SIZE   = 16   # T4 16 GB 安全；可尝试 24 或 32 on L4/A100
EVAL_BS      = 32
NUM_WORKERS  = 2
VAL_SPLIT    = 0.2
SEED         = 42
WEIGHT_DECAY = 1e-4
USE_AMP      = True
PATIENCE     = 5    # Stage-2 早停 patience

S1_EP, S1_LR = 5,  1e-3
S2_EP, S2_LR = 20, 1e-4
S3_EP, S3_LR = 5,  5e-5

DATA_DIR = Path('/kaggle/input/kul-computer-vision-ga-2-2026')
OUT      = Path('/kaggle/working') / EXPERIMENT
CKPT, MET, FIG, PRED, SUB = (OUT/'checkpoints', OUT/'metrics',
                               OUT/'figures', OUT/'predictions', OUT/'submissions')
for d in (CKPT, MET, FIG, PRED, SUB): d.mkdir(parents=True, exist_ok=True)

BEST_CKPT  = CKPT / 'best_model.pth'
FINAL_CKPT = CKPT / 'final_model.pth'
THRESH_NPY = MET  / 'best_thresholds.npy'

LABELS = [
    'aeroplane','bicycle','bird','boat','bottle',
    'bus','car','cat','chair','cow',
    'diningtable','dog','horse','motorbike','person',
    'pottedplant','sheep','sofa','train','tvmonitor',
]
print('Output:', OUT)

In [ ]:
# ── 工具函数 ───────────────────────────────────────────────────────────────────
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

class AsymmetricLoss(nn.Module):
    def __init__(self, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); self.gn, self.gp, self.clip, self.eps = gn, gp, clip, eps
    def forward(self, logits, targets):
        p  = torch.sigmoid(logits)
        pn = (1 - p + self.clip).clamp(max=1.0)
        lp = torch.log(p.clamp(min=self.eps))
        ln = torch.log(pn.clamp(min=self.eps))
        loss = targets * lp + (1 - targets) * ln
        with torch.no_grad():
            w = targets*(1-p).pow(self.gp) + (1-targets)*p.pow(self.gn)
        return -(loss * w).mean()

_MEAN, _STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

def train_tfm(sz=IMG_SIZE):
    return T.Compose([
        T.Resize((sz,sz)), T.RandomHorizontalFlip(), T.RandomRotation(15),
        T.ColorJitter(0.3,0.3,0.2,0.05), T.ToTensor(), T.Normalize(_MEAN,_STD),
        T.RandomErasing(p=0.3, scale=(0.02,0.2)),
    ])

def val_tfm(sz=IMG_SIZE):
    return T.Compose([T.Resize((sz,sz)), T.ToTensor(), T.Normalize(_MEAN,_STD)])

class VOCDataset(Dataset):
    def __init__(self, df, data_dir, split='train', transform=None):
        self.df, self.data_dir, self.split = df, Path(data_dir), split
        self.transform = transform or val_tfm()
        self.has_labels = all(c in df.columns for c in LABELS)
        self.indices = list(df.index)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        img = Image.fromarray(np.load(self.data_dir/self.split/'img'/f'{self.split}_{idx}.npy'))
        img = self.transform(img)
        if self.has_labels:
            return img, torch.FloatTensor(self.df.loc[idx, LABELS].values.astype(float))
        return img, idx

In [ ]:
# ── 显存探测 ───────────────────────────────────────────────────────────────────
def memory_probe():
    if not torch.cuda.is_available(): print('No CUDA'); return
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    m = models.efficientnet_v2_s(weights=None)
    net = nn.Sequential(m.features, m.avgpool, nn.Flatten(1),
                        nn.Linear(FEAT_DIM,512), nn.ReLU(), nn.Linear(512,20)).to(device).train()
    x = torch.randn(BATCH_SIZE,3,IMG_SIZE,IMG_SIZE,device=device)
    y = torch.rand(BATCH_SIZE,20,device=device)
    opt = torch.optim.AdamW(net.parameters(),lr=1e-4)
    scaler = torch.amp.GradScaler('cuda')
    t0 = time.perf_counter()
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda'): loss = nn.BCEWithLogitsLoss()(net(x),y)
    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()/1024**3
    del net,x,y,opt,scaler,loss; gc.collect(); torch.cuda.empty_cache()
    print(f'Memory probe: batch={BATCH_SIZE}, peak={peak:.2f} GB, {time.perf_counter()-t0:.1f}s/step')

memory_probe()

In [ ]:
# ── 模型定义 ───────────────────────────────────────────────────────────────────
class EfficientNetV2SClassifier(nn.Module):
    """EfficientNet-V2-S backbone + 两层分类头，输出 20 个 logit。

    特征提取路径：model.features -> model.avgpool -> flatten(1) -> 1280 维
    """
    def __init__(self, num_classes=20, pretrained=True):
        super().__init__()
        m = models.efficientnet_v2_s(
            weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None)
        self.features = nn.Sequential(m.features, m.avgpool)
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(FEAT_DIM, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )
        for l in self.classifier.modules():
            if isinstance(l, nn.Linear):
                nn.init.kaiming_normal_(l.weight); nn.init.zeros_(l.bias)

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad_(False)

    def unfreeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad_(True)

    def n_trainable(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

# 验证输出形状
_probe = EfficientNetV2SClassifier(pretrained=False)
with torch.no_grad():
    _out = _probe(torch.randn(2, 3, IMG_SIZE, IMG_SIZE))
print('模型输出形状:', _out.shape)   # 期望 (2, 20)
del _probe, _out

In [ ]:
# ── 训练循环工具 ───────────────────────────────────────────────────────────────
pin = device.type == 'cuda'

def run_epoch(model, loader, crit, opt, train, scaler=None):
    model.train(train); total = 0.0
    with torch.set_grad_enabled(train):
        for imgs, labels in tqdm(loader, leave=False, desc='train' if train else 'val'):
            imgs, labels = imgs.to(device,non_blocking=True), labels.to(device,non_blocking=True)
            with torch.amp.autocast('cuda', enabled=USE_AMP and pin):
                loss = crit(model(imgs), labels)
            if train:
                opt.zero_grad(set_to_none=True)
                if scaler and scaler.is_enabled():
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                else:
                    loss.backward(); opt.step()
            total += loss.item() * len(imgs)
    return total / len(loader.dataset)

def train_stage(model, tr_dl, vl_dl, crit, lr, epochs, tag, best_loss, hist, patience=None):
    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and pin)
    no_imp = 0
    for ep in range(1, epochs+1):
        tl = run_epoch(model, tr_dl, crit, opt, True, scaler)
        vl = run_epoch(model, vl_dl, crit, None, False)
        sched.step()
        best = vl < best_loss
        if best: best_loss = vl; torch.save(model.state_dict(), BEST_CKPT); no_imp = 0
        else: no_imp += 1
        torch.save(model.state_dict(), CKPT/'last_model.pth')
        hist.append(dict(stage=tag,epoch=ep,train_loss=tl,val_loss=vl,best=best))
        print(f'[{tag}] {ep:2d}/{epochs} | train={tl:.4f} val={vl:.4f}' + (' ← best' if best else ''))
        if patience and no_imp >= patience:
            print(f'[{tag}] Early stop (patience={patience})')
            break
    return best_loss

In [ ]:
# ── 三阶段训练 ─────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR/'train'/'train_set.csv', index_col='Id')
tr_idx, vl_idx = train_test_split(range(len(df)), test_size=VAL_SPLIT, random_state=SEED)

kw = dict(num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=NUM_WORKERS>0)
tr_dl = DataLoader(VOCDataset(df.iloc[tr_idx], DATA_DIR, 'train', train_tfm()),
                   BATCH_SIZE, shuffle=True,  **kw)
vl_dl = DataLoader(VOCDataset(df.iloc[vl_idx], DATA_DIR, 'train', val_tfm()),
                   EVAL_BS,    shuffle=False, **kw)

model = EfficientNetV2SClassifier(pretrained=True).to(device)
crit  = AsymmetricLoss()
hist, best = [], float('inf')

print('=== Stage 1: head only ===')
model.freeze_backbone()
print(f'Trainable params: {model.n_trainable():,}')
best = train_stage(model, tr_dl, vl_dl, crit, S1_LR, S1_EP, 'S1', best, hist)

print('\n=== Stage 2: full fine-tune ===')
model.unfreeze_backbone()
print(f'Trainable params: {model.n_trainable():,}')
best = train_stage(model, tr_dl, vl_dl, crit, S2_LR, S2_EP, 'S2', best, hist, patience=PATIENCE)

print(f'\n=== Stage 3: all {len(df)} samples ===')
model.load_state_dict(torch.load(BEST_CKPT, map_location=device))
full_dl = DataLoader(VOCDataset(df, DATA_DIR, 'train', train_tfm()), BATCH_SIZE, shuffle=True, **kw)
opt3   = torch.optim.AdamW(model.parameters(), lr=S3_LR, weight_decay=WEIGHT_DECAY)
sched3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=S3_EP)
sc3    = torch.amp.GradScaler('cuda', enabled=USE_AMP and pin)
for ep in range(1, S3_EP+1):
    model.train(); tl = 0.0
    for imgs, labels in tqdm(full_dl, leave=False, desc='S3'):
        imgs, labels = imgs.to(device,non_blocking=True), labels.to(device,non_blocking=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP and pin): loss = crit(model(imgs), labels)
        opt3.zero_grad(set_to_none=True)
        sc3.scale(loss).backward(); sc3.step(opt3); sc3.update()
        tl += loss.item() * len(imgs)
    sched3.step()
    tl /= len(full_dl.dataset)
    print(f'[S3] {ep}/{S3_EP} | train={tl:.4f}')

torch.save(model.state_dict(), FINAL_CKPT)
pd.DataFrame(hist).to_csv(MET/'training_history.csv', index=False)
print('\n训练完成，最佳 val loss:', round(best,6))

In [ ]:
# ── 验证集评估：mAP + per-class 最优阈值 ──────────────────────────────────────
model.load_state_dict(torch.load(BEST_CKPT, map_location=device))
model.eval()
all_p, all_l = [], []
with torch.no_grad():
    for imgs, labels in tqdm(vl_dl, desc='Val eval'):
        with torch.amp.autocast('cuda', enabled=USE_AMP and pin):
            all_p.append(torch.sigmoid(model(imgs.to(device))).float().cpu().numpy())
        all_l.append(labels.numpy())
all_p = np.vstack(all_p); all_l = np.vstack(all_l)

map_score = average_precision_score(all_l, all_p, average='macro')
print(f'Val mAP: {map_score:.6f}')

rows, thresholds = [], np.zeros(len(LABELS))
for i, cls in enumerate(LABELS):
    ap = average_precision_score(all_l[:,i], all_p[:,i])
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.1, 0.91, 0.05):
        preds = (all_p[:,i] > t).astype(int)
        tp = (preds * all_l[:,i]).sum()
        f1 = 2*tp / (2*tp + (preds*(1-all_l[:,i])).sum() + ((1-preds)*all_l[:,i]).sum() + 1e-8)
        if f1 > best_f1: best_f1, best_t = f1, t
    thresholds[i] = best_t
    rows.append(dict(cls=cls, ap=round(ap,4), threshold=best_t, f1=round(best_f1,4)))
    print(f'{cls:<14s} AP={ap:.4f}  thr={best_t:.2f}  F1={best_f1:.3f}')

np.save(THRESH_NPY, thresholds)
ap_df = pd.DataFrame(rows)
ap_df.to_csv(MET/'ap_per_class.csv', index=False)
pd.DataFrame([dict(experiment=EXPERIMENT, mAP=map_score)]).to_csv(MET/'evaluation_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(12,4))
ax.bar(ap_df['cls'], ap_df['ap'], color='#5ba85a')
ax.axhline(map_score, color='red', linestyle='--', lw=1.4, label=f'mAP={map_score:.3f}')
ax.set_ylim(0,1.05); ax.set_ylabel('AP'); ax.tick_params(axis='x',rotation=45)
for tick in ax.get_xticklabels(): tick.set_ha('right')
ax.legend(); ax.set_title(f'Per-class AP ({EXPERIMENT})')
fig.tight_layout()
fig.savefig(FIG/'eval_ap_per_class.png', dpi=140); plt.show()

In [ ]:
# ── 测试集推理（TTA）+ 生成 1500 行提交 CSV ────────────────────────────────────
def rle_encode(arr):
    pixels = np.concatenate([[0], arr.flatten(), [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

ckpt = FINAL_CKPT if FINAL_CKPT.exists() else BEST_CKPT
model.load_state_dict(torch.load(ckpt, map_location=device))
model.eval()
thresholds = np.load(THRESH_NPY)
print('Using:', ckpt)

test_df = pd.read_csv(DATA_DIR/'test'/'test_set.csv', index_col='Id')
test_ds = VOCDataset(test_df, DATA_DIR, 'test', val_tfm())
test_dl = DataLoader(test_ds, EVAL_BS, shuffle=False, **kw)

all_probs = []
with torch.no_grad():
    for imgs, _ in tqdm(test_dl, desc='Test TTA'):
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP and pin):
            p = (torch.sigmoid(model(imgs)) + torch.sigmoid(model(imgs.flip(-1)))) / 2
        all_probs.append(p.float().cpu().numpy())

all_probs = np.vstack(all_probs)
all_ids   = list(test_ds.indices)
assert len(all_probs) == len(all_ids), f'{len(all_probs)} vs {len(all_ids)}'
preds = (all_probs > thresholds[None,:]).astype(int)

prob_df = pd.DataFrame(all_probs, columns=LABELS, index=all_ids); prob_df.index.name='Id'
pred_df = pd.DataFrame(preds,     columns=LABELS, index=all_ids); pred_df.index.name='Id'
prob_df.to_csv(PRED/f'test_probabilities_{EXPERIMENT}.csv')
pred_df.to_csv(PRED/f'test_binary_predictions_{EXPERIMENT}.csv')

rows = {'Id':[], 'Predicted':[]}
for idx in pred_df.index:
    rows['Id'].append(f'{idx}_classification')
    rows['Predicted'].append(rle_encode(pred_df.loc[idx, LABELS].values.astype(int)))
    rows['Id'].append(f'{idx}_segmentation')
    rows['Predicted'].append('')
sub = pd.DataFrame(rows).set_index('Id')
sub_path = SUB / f'submission_classification_{EXPERIMENT}.csv'
sub.to_csv(sub_path)
print(f'提交文件: {sub_path} ({len(sub)} 行)')
sub.head(4)